# Notebook 1 — Preparación de Datos (CRISP-DM)
## Plataforma *DatosParaTodos* · datos.gov.co

Este notebook cubre las **tres primeras fases de CRISP-DM**:

| Fase | Descripción |
|------|-------------|
| 1. Comprensión del Negocio | Definición del problema, preguntas analíticas, KPIs |
| 2. Comprensión de los Datos | EDA descriptivo e inferencial, correlaciones, outliers |
| 3. Preparación de los Datos | Pipeline de limpieza, imputación, normalización, ingeniería de características |

**Fuente de datos:** API SODA pública de [datos.gov.co](https://www.datos.gov.co) — Dataset de Accidentalidad Vial en Bogotá (`vjvu-ycr3`).

---

## Fase 1 — Comprensión del Negocio (Business Understanding)

### Contexto del problema

La **accidentalidad vial** es uno de los principales problemas de salud pública en Colombia. Bogotá registra miles de incidentes de tránsito anuales, con consecuencias que van desde daños materiales hasta víctimas fatales. El Observatorio de Movilidad de la ciudad publica estos datos en el portal de datos abiertos para permitir su análisis.

### Pregunta analítica principal

> **¿Qué factores determinan la gravedad de un accidente de tránsito en Bogotá?**

Esta pregunta tiene valor operativo directo: si un modelo puede predecir la gravedad esperada a partir de condiciones conocidas (hora, tipo de vehículo, localidad, clase de accidente), la Secretaría de Movilidad puede priorizar recursos de atención y diseñar intervenciones preventivas focalizadas.

### Preguntas de negocio derivadas

1. ¿En qué horas del día se concentran los accidentes más graves?
2. ¿Qué clase de accidente (choque, atropello, volcamiento) produce más víctimas?
3. ¿Existe correlación entre el estrato socioeconómico de la zona y la gravedad?
4. ¿Se pueden identificar localidades con perfil de accidentalidad diferenciado?
5. ¿Con qué precisión se puede clasificar automáticamente la gravedad de un nuevo incidente?

### Definición de la variable objetivo

La variable **`gravedad`** categoriza el resultado del accidente:
- `SOLO DAÑOS` — Sin víctimas, únicamente daños materiales
- `CON HERIDOS` — Al menos un herido no fatal
- `CON MUERTOS` — Al menos una víctima fatal

### Métrica de éxito

| KPI | Umbral mínimo |
|-----|---------------|
| F1-Score ponderado | ≥ 0.70 |
| AUC-ROC (OvR) | ≥ 0.75 |
| Accuracy en test | ≥ 0.65 |

## Fase 2 — Comprensión de los Datos (Data Understanding)

### 1. Instalación e importación de dependencias

In [ ]:
!pip install pandas numpy requests ydata-profiling scikit-learn matplotlib seaborn --quiet

import json
import math
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

try:
    from ydata_profiling import ProfileReport
    HAS_PROFILING = True
except ImportError:
    HAS_PROFILING = False
    print('ydata-profiling no disponible, se omiten reportes automáticos.')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.decomposition import PCA

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
PALETTE = sns.color_palette('tab10')
DATE_PATTERN = re.compile(r'\d{4}-\d{2}')

print('Dependencias cargadas correctamente.')

### 2. Ingesta de datos reales — datos.gov.co

Descargamos el dataset de **Accidentalidad Vial en Bogotá** a través de la API SODA pública.  
El dataset contiene registros de incidentes con variables de tiempo, espacio y resultado del accidente.

In [ ]:
URL = 'https://www.datos.gov.co/resource/vjvu-ycr3.json?$limit=5000'

response = requests.get(URL, timeout=30)
response.raise_for_status()
raw_data = response.json()

df_raw = pd.DataFrame(raw_data)

print(f'Registros descargados : {len(df_raw):,}')
print(f'Variables             : {df_raw.shape[1]}')
print(f'Columnas              : {df_raw.columns.tolist()}')
df_raw.head(3)

### 3. Exploración inicial del dataset

Antes de cualquier transformación, exploramos la estructura del dato crudo para entender qué tenemos: tipos de datos, completitud, y estadísticas descriptivas básicas.

In [ ]:
print('=== TIPOS DE DATOS ===')
print(df_raw.dtypes.to_string())

print(f'\n=== NULOS POR COLUMNA ===')
nulos = df_raw.isnull().sum()
pct_nulos = (nulos / len(df_raw) * 100).round(1)
nulos_df = pd.DataFrame({'Nulos': nulos, '% del total': pct_nulos})
print(nulos_df[nulos_df['Nulos'] > 0].to_string() if nulos.sum() > 0 else 'No se detectaron nulos en el dataset crudo.')

print(f'\n=== ESTADÍSTICAS DESCRIPTIVAS ===')
df_raw.describe(include='all').T

### 4. Pandas Profiling — Exploración automática de la calidad del dato

Generamos el reporte completo con distribuciones, correlaciones, valores faltantes y alertas **antes** de limpiar, para entender el estado inicial del dataset.

In [ ]:
if HAS_PROFILING:
    profile = ProfileReport(
        df_raw,
        title='DatosParaTodos — Reporte de Calidad Inicial',
        explorative=True,
        minimal=False
    )
    profile.to_widgets()
else:
    print('ydata-profiling no disponible. Continuamos con el EDA manual.')

### 5. EDA — Análisis Exploratorio de Datos con Interpretación

#### 5.1 Distribución de la variable objetivo

La distribución de `gravedad` es el primer diagnóstico crítico: define si el problema es de clasificación balanceada o desbalanceada, y condiciona las decisiones de modelado (uso de SMOTE, elección de métrica principal).

In [ ]:
TARGET_CANDIDATES = ['gravedad', 'clase_accidente', 'clase_bien', 'tipo_accidente', 'resultado']
target_col_eda = None
for col in TARGET_CANDIDATES:
    if col in df_raw.columns:
        target_col_eda = col
        break
if target_col_eda is None:
    for col in df_raw.columns:
        vals = df_raw[col].dropna()
        n_u = vals.nunique()
        if 2 <= n_u <= 10:
            target_col_eda = col
            break

if target_col_eda:
    vc = df_raw[target_col_eda].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    vc.plot(kind='bar', ax=axes[0], color=PALETTE[:len(vc)], edgecolor='black', linewidth=0.7)
    axes[0].set_title(f'Distribución de clases — {target_col_eda}', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Frecuencia')
    axes[0].tick_params(axis='x', rotation=30)
    for bar in axes[0].patches:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                     str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

    axes[1].pie(vc.values, labels=vc.index, autopct='%1.1f%%', colors=PALETTE[:len(vc)],
                startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    axes[1].set_title(f'Proporción relativa — {target_col_eda}', fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.show()

    ratio = vc.min() / vc.max()
    print(f'=== Análisis de balanceo ===')
    print(f'Clases detectadas        : {vc.index.tolist()}')
    print(f'Conteos                  : {vc.values.tolist()}')
    print(f'Ratio min/max            : {ratio:.2f}')
    print(f'Desbalanceo severo (<0.3): {"SÍ — se aplicará SMOTE en entrenamiento" if ratio < 0.3 else "NO — distribución aceptable"}')
    print(f'\n📌 INTERPRETACIÓN:')
    print(f'   La clase mayoritaria representa el {vc.values[0]/vc.sum()*100:.1f}% de los casos.')
    if ratio < 0.5:
        print(f'   El desbalance es significativo. Un clasificador que prediga siempre la clase mayoritaria')
        print(f'   obtendría {vc.values[0]/vc.sum()*100:.1f}% de accuracy sin aprender nada.')
        print(f'   Por esto usamos F1-Score y AUC-ROC como métricas principales, no accuracy.')
else:
    print('No se encontró una variable objetivo clara.')
    print('Columnas disponibles:', df_raw.columns.tolist())

#### 5.2 Accidentalidad por franja horaria

La hora del accidente es una de las variables con mayor poder predictivo en la literatura de seguridad vial. Se esperan picos en horas punta (7-9h, 17-20h) y patrones de mayor severidad en horas nocturnas.

In [ ]:
HORA_COLS = ['hora', 'hora_dia', 'hora_accidente', 'hora_ocurrencia']
hora_col = next((c for c in HORA_COLS if c in df_raw.columns), None)

if hora_col:
    df_hora = df_raw[hora_col].dropna().astype(str)
    horas = pd.to_numeric(df_hora.str.extract(r'(\d{1,2})')[0], errors='coerce').dropna()
    horas = horas[(horas >= 0) & (horas <= 23)].astype(int)

    fig, ax = plt.subplots(figsize=(12, 5))
    conteo = horas.value_counts().sort_index()
    ax.bar(conteo.index, conteo.values, color=PALETTE[0], alpha=0.8, edgecolor='black', linewidth=0.5)

    for hora_punta, label in [(8, 'Punta\nmañana'), (18, 'Punta\ntarde')]:
        ax.axvline(hora_punta, color='red', linestyle='--', alpha=0.6, linewidth=1.5)
        ax.text(hora_punta + 0.3, conteo.max() * 0.92, label, color='red', fontsize=9)

    ax.set_title('Distribución de accidentes por hora del día', fontsize=13, fontweight='bold')
    ax.set_xlabel('Hora del día (0 = medianoche)')
    ax.set_ylabel('Número de accidentes')
    ax.set_xticks(range(0, 24))
    plt.tight_layout()
    plt.show()

    hora_pico = conteo.idxmax()
    pct_nocturno = horas[(horas >= 22) | (horas <= 5)].count() / len(horas) * 100
    pct_punta = horas[((horas >= 7) & (horas <= 9)) | ((horas >= 17) & (horas <= 20))].count() / len(horas) * 100

    print(f'📌 INTERPRETACIÓN:')
    print(f'   Hora con más accidentes : {hora_pico}:00h ({conteo[hora_pico]} incidentes)')
    print(f'   Accidentes en horas punta (7-9h, 17-20h): {pct_punta:.1f}%')
    print(f'   Accidentes nocturnos (22h-5h)           : {pct_nocturno:.1f}%')
    print(f'   Las horas punta concentran tráfico denso → mayor frecuencia de choques.')
    print(f'   Las horas nocturnas tienen menor densidad pero velocidades más altas → mayor severidad.')
    print(f'   La hora es una variable de alta importancia para el modelo predictivo.')
else:
    print('Columna de hora no encontrada. Columnas disponibles:', df_raw.columns.tolist())

#### 5.3 Distribución por clase/tipo de accidente

Diferentes modalidades (choque, atropello, caída de ocupante, volcamiento) tienen perfiles de gravedad distintos. Esta variable es clave para EDA y para el modelo.

In [ ]:
CLASE_COLS = ['clase_accidente', 'clase', 'tipo_accidente', 'modalidad']
clase_col = next((c for c in CLASE_COLS if c in df_raw.columns and c != target_col_eda), None)

if clase_col:
    vc_clase = df_raw[clase_col].value_counts().head(10)

    fig, ax = plt.subplots(figsize=(11, 5))
    colors_bar = [PALETTE[i % len(PALETTE)] for i in range(len(vc_clase))]
    vc_clase.sort_values().plot(kind='barh', ax=ax, color=colors_bar, edgecolor='black', linewidth=0.5)
    ax.set_title(f'Tipos de accidente más frecuentes — {clase_col}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Frecuencia')

    for i, (v, p) in enumerate(zip(vc_clase.sort_values().values, vc_clase.sort_values().values / vc_clase.sum() * 100)):
        ax.text(v + 2, i, f'{p:.1f}%', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    tipo_comun = vc_clase.index[0]
    pct_comun = vc_clase.values[0] / vc_clase.sum() * 100
    print(f'📌 INTERPRETACIÓN:')
    print(f'   El tipo de accidente más común es "{tipo_comun}" ({pct_comun:.1f}% del total).')
    print(f'   Los choques dominan en ciudades con alta densidad vehicular como Bogotá.')
    print(f'   Los atropellos, aunque menos frecuentes, tienen mayor gravedad promedio.')
    print(f'   Esta variable es un predictor fuerte de la gravedad del resultado.')
elif target_col_eda:
    print(f'Mostrando la variable objetivo "{target_col_eda}" (ya visualizada en 5.1).')
else:
    print('Columna de clase/tipo de accidente no encontrada.')

#### 5.4 Distribución geográfica — Accidentes por localidad

La localidad captura el contexto urbano: infraestructura vial, densidad de tráfico, presencia de zonas escolares o industriales. Se espera heterogeneidad significativa entre localidades.

In [ ]:
LOC_COLS = ['localidad', 'localidad_accidente', 'barrio', 'upz', 'zona']
loc_col = next((c for c in LOC_COLS if c in df_raw.columns), None)

if loc_col:
    vc_loc = df_raw[loc_col].value_counts().head(12)

    fig, ax = plt.subplots(figsize=(12, 6))
    colors_loc = [PALETTE[i % len(PALETTE)] for i in range(len(vc_loc))]
    vc_loc.sort_values().plot(kind='barh', ax=ax, color=colors_loc, edgecolor='black', linewidth=0.5)
    ax.set_title(f'Accidentes por localidad/zona (Top 12) — {loc_col}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Número de accidentes')

    for i, v in enumerate(vc_loc.sort_values().values):
        ax.text(v + 2, i, str(v), va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    top3_loc = vc_loc.head(3).index.tolist()
    pct_top3 = vc_loc.head(3).sum() / df_raw[loc_col].count() * 100
    print(f'📌 INTERPRETACIÓN:')
    print(f'   Top 3 localidades con más accidentes: {top3_loc}')
    print(f'   El Top 3 concentra el {pct_top3:.1f}% de todos los accidentes registrados.')
    print(f'   Estas zonas suelen corresponder a ejes viales principales y zonas comerciales.')
    print(f'   La localidad tiene alto valor predictivo y debe incluirse en el modelo.')
else:
    print('Columna de localidad/zona no encontrada. Columnas disponibles:', df_raw.columns.tolist())

#### 5.5 Variables numéricas — Distribuciones y correlaciones

Visualizamos las distribuciones de las variables numéricas continuas para detectar asimetría, multimodalidad y potenciales outliers antes de la limpieza formal.

In [ ]:
def detect_columns_raw(df):
    sample = df.head(50)
    numeric, date, categorical = [], [], []
    for col in df.columns:
        vals = sample[col].dropna().tolist()
        if not vals:
            categorical.append(col); continue
        num_count  = sum(1 for v in vals if pd.notna(pd.to_numeric(v, errors='coerce')))
        date_count = sum(1 for v in vals if isinstance(v, str) and DATE_PATTERN.search(v))
        if num_count > len(vals) * 0.6:
            numeric.append(col)
        elif date_count > len(vals) * 0.5:
            date.append(col)
        else:
            categorical.append(col)
    return {'numeric': numeric, 'date': date, 'categorical': categorical}

col_types_raw = detect_columns_raw(df_raw)
num_cols_raw  = col_types_raw['numeric']

df_num = df_raw[num_cols_raw].apply(pd.to_numeric, errors='coerce') if num_cols_raw else pd.DataFrame()

if len(num_cols_raw) > 0:
    n_plots = min(len(num_cols_raw), 6)
    cols_plot = num_cols_raw[:n_plots]
    ncols_grid = min(3, n_plots)
    nrows_grid = math.ceil(n_plots / ncols_grid)
    fig, axes = plt.subplots(nrows_grid, ncols_grid, figsize=(5 * ncols_grid, 4 * nrows_grid))
    axes = np.array(axes).flatten()

    for i, col in enumerate(cols_plot):
        data = df_num[col].dropna()
        axes[i].hist(data, bins=30, color=PALETTE[i % len(PALETTE)], alpha=0.8, edgecolor='white')
        skew = data.skew()
        axes[i].set_title(col, fontsize=11)
        axes[i].set_xlabel(f'Sesgo: {skew:.2f}', fontsize=9, color='gray')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Distribuciones de variables numéricas (datos crudos)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('📌 INTERPRETACIÓN DE ASIMETRÍA:')
    for col in cols_plot:
        skew = df_num[col].skew()
        if skew > 1:
            desc = 'sesgada a la derecha — posibles outliers altos'
        elif skew < -1:
            desc = 'sesgada a la izquierda — posibles outliers bajos'
        elif abs(skew) > 0.5:
            desc = 'moderadamente asimétrica'
        else:
            desc = 'aproximadamente simétrica'
        print(f'   {col}: {desc} (sesgo={skew:.2f})')
else:
    print('No se detectaron columnas numéricas en el dataset crudo.')

#### 5.6 Matriz de correlación (pre-limpieza)

La correlación de Pearson entre variables numéricas revela relaciones lineales. Valores |r| > 0.90 indican redundancia — esas variables serán eliminadas en el pipeline.

In [ ]:
if len(num_cols_raw) >= 2:
    corr_raw = df_num.corr()
    mask = np.triu(np.ones_like(corr_raw, dtype=bool))

    fig, ax = plt.subplots(figsize=(max(8, len(num_cols_raw)), max(6, len(num_cols_raw) - 1)))
    sns.heatmap(corr_raw, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                vmin=-1, vmax=1, center=0, linewidths=0.5,
                annot_kws={'size': 9}, ax=ax)
    ax.set_title('Matriz de correlación de Pearson — Variables numéricas crudas', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    corr_abs = corr_raw.abs()
    upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
    pares_altos = [(c1, c2, round(upper.loc[c1, c2], 3))
                   for c1 in upper.index for c2 in upper.columns
                   if pd.notna(upper.loc[c1, c2]) and upper.loc[c1, c2] > 0.7]

    if pares_altos:
        print('📌 Pares con correlación alta (|r| > 0.7):')
        for c1, c2, r in sorted(pares_altos, key=lambda x: -x[2]):
            marker = '  ⚠️  REDUNDANTE — se eliminará en pipeline' if r > 0.9 else ''
            print(f'   {c1} ↔ {c2}: r = {r}{marker}')
    else:
        print('📌 No se encontraron correlaciones altas. Todas las variables aportan información independiente.')
else:
    print('Se requieren al menos 2 columnas numéricas para la matriz de correlación.')

---

## Fase 3 — Preparación de los Datos (Data Preparation)

El pipeline de limpieza implementado en `app/services/analytics.py` sigue 6 pasos ordenados:

1. Deduplicación
2. Imputación de nulos (mediana / moda)
3. Corrección de outliers (IQR winsorization)
4. Normalización de formatos
5. Eliminación de varianza cero
6. Eliminación de redundancias (Pearson > 0.90)

### 6. Detección automática de tipos de columna

In [ ]:
def detect_columns(df):
    """Replica exacta de AnalyticsService.detect_columns() con detección ordinal."""
    sample = df.head(50)
    numeric, date, categorical = [], [], []
    for col in df.columns:
        vals = sample[col].dropna().tolist()
        if not vals:
            categorical.append(col); continue
        num_count  = sum(1 for v in vals if pd.notna(pd.to_numeric(v, errors='coerce')))
        date_count = sum(1 for v in vals if isinstance(v, str) and DATE_PATTERN.search(v))
        if num_count > len(vals) * 0.6:
            floats = [float(pd.to_numeric(v, errors='coerce')) for v in vals
                      if pd.notna(pd.to_numeric(v, errors='coerce'))]
            all_integers = all(f % 1 == 0 for f in floats)
            n_unique = len(set(floats))
            if all_integers and n_unique <= 15:
                categorical.append(col)  # ordinal → OHE
            else:
                numeric.append(col)
        elif date_count > len(vals) * 0.5:
            date.append(col)
        else:
            categorical.append(col)
    return {'numeric': numeric, 'date': date, 'categorical': categorical}

col_types = detect_columns(df_raw)
print(f"Columnas numéricas   ({len(col_types['numeric'])}): {col_types['numeric']}")
print(f"Columnas fecha       ({len(col_types['date'])}): {col_types['date']}")
print(f"Columnas categóricas ({len(col_types['categorical'])}): {col_types['categorical']}")

print('\n=== Variables ordinales detectadas ===')
print('(Parecen numéricas pero tienen ≤15 valores únicos enteros → se tratan como categóricas con OHE):')
ordinales_encontradas = []
for col in col_types['categorical']:
    vals = df_raw[col].dropna().tolist()[:50]
    nums = [pd.to_numeric(v, errors='coerce') for v in vals]
    nums_ok = [n for n in nums if pd.notna(n)]
    if nums_ok and len(nums_ok) > len(vals) * 0.6:
        uniq = sorted(set(nums_ok))
        if all(v % 1 == 0 for v in uniq) and len(uniq) <= 15:
            print(f'   {col}: valores = {[int(u) for u in uniq]}')
            ordinales_encontradas.append(col)
if not ordinales_encontradas:
    print('   (ninguna — el dataset no tiene variables tipo estrato/mes en este caso)')

### 7. Eliminación de registros duplicados

In [ ]:
n_antes = len(df_raw)
df = df_raw.drop_duplicates().reset_index(drop=True)
n_dup = n_antes - len(df)

print(f'Registros antes : {n_antes:,}')
print(f'Duplicados      : {n_dup:,} ({n_dup/n_antes*100:.1f}%)')
print(f'Registros tras  : {len(df):,}')
print(f'\n📌 Los duplicados sesgarían estadísticas y el entrenamiento: el mismo evento')
print(f'   tendría más peso que otros. Se eliminan como primer paso obligatorio.')

### 8. Imputación de valores nulos

In [ ]:
nulos_antes = df.isnull().sum()
total_nulos = nulos_antes.sum()

if total_nulos > 0:
    print(f'Total de celdas nulas: {total_nulos:,}')
    pct = (nulos_antes / len(df) * 100).round(1)
    print(pd.DataFrame({'Nulos': nulos_antes[nulos_antes > 0], '%': pct[nulos_antes > 0]}).to_string())
else:
    print('No hay valores nulos.')

medianas = {}
for col in col_types['numeric']:
    if col in df.columns:
        vals = pd.to_numeric(df[col], errors='coerce').dropna()
        medianas[col] = float(vals.median()) if len(vals) > 0 else 0.0
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(medianas[col])

for col in col_types['categorical']:
    if col in df.columns:
        df[col] = df[col].fillna('No especificado')

print(f'\n✓ Nulos restantes: {df.isnull().sum().sum()}')
print(f'\n📌 La mediana es más robusta que la media ante outliers.')
print(f'   "No especificado" preserva la información de que el dato faltaba.')

### 9. Detección y corrección de outliers — Método IQR

Para columnas numéricas con más de 15 valores únicos:
- `lower = Q1 - 1.5 × IQR`, `upper = Q3 + 1.5 × IQR`
- Valores fuera del rango se **capan** (*winsorization*) — se preserva el registro, se ajusta el valor al límite plausible.

**Por qué no eliminar:** en accidentalidad vial, un valor extremo puede ser real (accidente masivo). Capearlo es más honesto que borrarlo.

In [ ]:
outlier_log = []
bounds = {}

for col in col_types['numeric']:
    if col not in df.columns: continue
    vals_sorted = df[col].dropna().sort_values().tolist()
    n = len(vals_sorted)
    if n < 4 or len(set(vals_sorted)) < 15: continue

    q1 = vals_sorted[int(n * 0.25)]
    q3 = vals_sorted[int(n * 0.75)]
    iqr = q3 - q1
    if iqr == 0: continue

    low  = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    bounds[col] = (low, high)

    mask_low  = df[col] < low
    mask_high = df[col] > high
    n_out = mask_low.sum() + mask_high.sum()

    if n_out > 0:
        df.loc[mask_low,  col] = round(low, 2)
        df.loc[mask_high, col] = round(high, 2)
        outlier_log.append({
            'Columna': col, 'Outliers': int(n_out),
            '% total': round(n_out / n * 100, 1),
            'Lím. inferior': round(low, 2), 'Lím. superior': round(high, 2)
        })

if outlier_log:
    df_out = pd.DataFrame(outlier_log)
    display(df_out)
    print(f'\n📌 Total de valores modificados: {df_out["Outliers"].sum()}')
    print(f'   IQR es no-paramétrico: no asume distribución normal.')
else:
    print('✓ No se detectaron outliers con el criterio IQR.')

### Visualización: Distribuciones antes vs. después del tratamiento de outliers

In [ ]:
cols_con_outliers = [c for c in col_types['numeric'] if c in df.columns and c in bounds][:4]

if cols_con_outliers:
    n = len(cols_con_outliers)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for i, col in enumerate(cols_con_outliers):
        data_antes   = pd.to_numeric(df_raw[col], errors='coerce').dropna()
        data_despues = df[col].dropna()
        axes[0][i].hist(data_antes, bins=30, color='#e74c3c', alpha=0.7, edgecolor='white')
        axes[0][i].set_title(f'{col}\nANTES (sesgo={data_antes.skew():.2f})', fontsize=10)
        axes[1][i].hist(data_despues, bins=30, color='#27ae60', alpha=0.7, edgecolor='white')
        axes[1][i].set_title(f'{col}\nDESPUÉS (sesgo={data_despues.skew():.2f})', fontsize=10)

    plt.suptitle('Efecto de la corrección de outliers por IQR', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No hay columnas con outliers para graficar.')

### 10. Normalización de formatos

In [ ]:
for col in col_types['numeric']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

for col in col_types['categorical']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()

print('✓ Formatos normalizados: numéricos → float64, categóricos → MAYÚSCULAS')
print('  Consolida: "choque", "Choque", "CHOQUE" → "CHOQUE"')
df.head(3)

### 11. Eliminación de variables con varianza cero

In [ ]:
nunique = df.nunique(dropna=True)
cols_constantes = nunique[nunique <= 1].index.tolist()

if cols_constantes:
    print(f'Eliminando {len(cols_constantes)} columnas constantes: {cols_constantes}')
    print('📌 Una variable constante no discrimina entre clases.')
    df = df.drop(columns=cols_constantes)
else:
    print('✓ No hay columnas con varianza cero.')

print(f'Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas')

### 12. Eliminación de variables redundantes (Pearson > 0.90)

In [ ]:
num_cols_final = [c for c in col_types['numeric'] if c in df.columns]
redundant_cols = []

if len(num_cols_final) >= 2:
    corr_matrix = df[num_cols_final].corr(numeric_only=True).abs()
    mask_upper  = np.triu(np.ones_like(corr_matrix, dtype=bool))

    fig, ax = plt.subplots(figsize=(max(7, len(num_cols_final)), max(5, len(num_cols_final) - 1)))
    sns.heatmap(corr_matrix, mask=mask_upper, annot=True, fmt='.2f', cmap='RdYlGn_r',
                vmin=0, vmax=1, linewidths=0.5, ax=ax,
                cbar_kws={'label': '|Correlación de Pearson|'})
    ax.set_title('Correlaciones tras limpieza — Variables numéricas', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    redundant_cols = [col for col in upper.columns if any(upper[col] > 0.90)]

    if redundant_cols:
        df = df.drop(columns=redundant_cols)
        print(f'✓ Variables redundantes eliminadas (|r|>0.90): {redundant_cols}')
    else:
        print('✓ No hay variables redundantes (ningún par supera |r|=0.90).')
else:
    print('Menos de 2 columnas numéricas — análisis de correlación no aplicable.')

### 13. Ingeniería de características — Pipeline sklearn

| Tipo de variable | Transformación | Justificación |
|-----------------|---------------|---------------|
| Numérico (modelos lineales) | Imputer → StandardScaler → PCA(95%) | Sensibles a escala; PCA reduce dimensionalidad |
| Numérico (árboles) | Imputer → KBinsDiscretizer | Invariantes a escala; se benefician de discretización |
| Categórico (todos) | Imputer → OneHotEncoder | Convierte categorías sin orden implícito |

In [ ]:
cat_cols_all = [c for c in col_types['categorical'] if c in df.columns]
num_cols_all = [c for c in col_types['numeric']     if c in df.columns]

target_col = None
for col in cat_cols_all:
    n_u = df[col].nunique()
    if 2 <= n_u <= 20:
        target_col = col
        break

assert target_col, 'No se detectó variable objetivo.'
df = df.dropna(subset=[target_col])

y_raw    = df[target_col].astype(str)
X_df     = df.drop(columns=[target_col])
num_cols = [c for c in num_cols_all if c != target_col]
cat_cols = [c for c in cat_cols_all if c != target_col]

print(f'Variable objetivo  : "{target_col}"')
print(f'Clases             : {sorted(y_raw.unique().tolist())}')
print(f'Columnas numéricas : {num_cols}')
print(f'Columnas categóricas: {cat_cols}')

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('pca',     PCA(n_components=0.95, random_state=42))
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

transformers = []
if num_cols: transformers.append(('num', numeric_transformer, num_cols))
if cat_cols: transformers.append(('cat', categorical_transformer, cat_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

le = LabelEncoder()
y  = le.fit_transform(y_raw)
X_processed = preprocessor.fit_transform(X_df)

print(f'\nMatriz X procesada : {X_processed.shape}')
print(f'Clases codificadas : {list(enumerate(le.classes_))}')

if num_cols:
    pca_step = preprocessor.named_transformers_.get('num', {}.get('pca'))
    try:
        pca_obj = preprocessor.named_transformers_['num'].named_steps['pca']
        var_exp = pca_obj.explained_variance_ratio_
        print(f'PCA: {len(num_cols)} variables numéricas → {len(var_exp)} componentes ({var_exp.sum()*100:.1f}% varianza)')
    except:
        pass

### 14. Score de calidad del dato y conclusiones

In [ ]:
total_cells   = df.shape[0] * df.shape[1]
nulos_finales = df.isnull().sum().sum()
completitud   = max(0.0, 100.0 - (nulos_finales / total_cells * 100.0))
filas_unicas  = df.drop_duplicates().shape[0]
unicidad      = min(100.0, filas_unicas / len(df) * 100.0)
score         = round(completitud * 0.6 + unicidad * 0.4)

fig, ax = plt.subplots(figsize=(7, 4))
metricas_score = {'Completitud\n(60%)': completitud, 'Unicidad\n(40%)': unicidad, 'Score\nFinal': score}
bars = ax.bar(metricas_score.keys(), metricas_score.values(),
              color=['#3498db', '#2ecc71', '#e67e22'], edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, metricas_score.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}/100', ha='center', fontweight='bold', fontsize=13)
ax.set_ylim(0, 125)
ax.set_ylabel('Puntuación')
ax.set_title('Score de calidad del dataset limpio', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('=' * 55)
print(f'  Completitud              : {completitud:.1f}%')
print(f'  Unicidad                 : {unicidad:.1f}%')
print(f'  SCORE FINAL DE CALIDAD   : {score}/100')
print('=' * 55)

print(f'\n📌 CONCLUSIONES DEL PIPELINE DE PREPARACIÓN:')
print(f'   • Dataset final: {df.shape[0]:,} registros × {df.shape[1]} variables')
print(f'   • Dimensionalidad X para modelado: {X_processed.shape[1]} dimensiones')
print(f'   • Variable objetivo: "{target_col}" con {len(le.classes_)} clases')
print(f'   • Clases: {le.classes_.tolist()}')
print(f'   • Duplicados eliminados: {n_dup}')
if outlier_log:
    total_out = sum(o["Outliers"] for o in outlier_log)
    print(f'   • Outliers corregidos: {total_out} valores en {len(outlier_log)} columnas')
if redundant_cols:
    print(f'   • Variables redundantes eliminadas: {redundant_cols}')
print(f'\n   El dataset está listo para Notebook 2 — Modelado y Evaluación.')

df.describe(include='all')

---

## Resumen ejecutivo — Fases CRISP-DM completadas

| Paso | Acción | Resultado |
|------|--------|-----------|
| Business Understanding | Definición del problema, KPIs, pregunta analítica | Marco de evaluación establecido |
| EDA inicial | Profiling, distribuciones, correlaciones | Dominio comprendido |
| Deduplicación | Eliminación de filas exactas | Sesgo por repetición eliminado |
| Imputación | Mediana / "No especificado" | Completitud 100% |
| Outliers | IQR winsorización | Valores extremos corregidos |
| Normalización | MAYÚSCULAS, float64 | Categorías consolidadas |
| Varianza cero | Drop de columnas constantes | Variables informativas |
| Correlación | Pearson > 0.90 → eliminar | Multicolinealidad reducida |
| Feature Eng. | PCA + StandardScaler + OHE | X lista para modelado |

**Continúa en:** `2_Modelado_y_Evaluacion.ipynb`